# 01 — 图谱构建流水线端到端测试

覆盖 backend/graph 模块所有 6 个阶段。

**前置条件**：Neo4j 已启动，环境变量已配置。
**注意**：Stage 4 需要 Neo4j GDS 插件，Stage 2/4 需要 LLM API Key。

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from backend.pipelines.file_reader import FileReader
from backend.pipelines.text_chunker import ChineseTextChunker
from backend.pipelines.models import Document, Chunk
from backend.graph.core import connection_manager, get_connection_manager
from backend.graph.structure import GraphStructureBuilder
from backend.graph.extraction import EntityRelationExtractor, GraphWriter
from backend.graph.indexing import ChunkIndexManager, EntityIndexManager
from backend.graph.processing import (
    SimilarEntityDetector, GDSConfig,
    EntityMerger, EntityQualityProcessor
)
from backend.graph.graph_consistency_validator import GraphConsistencyValidator
from backend.models.get_models import get_llm_model
from backend.config.settings import FILES_DIR
from backend.config.prompts.graph_prompts import (
    system_template_build_graph, human_template_build_graph,
)
from backend.config.settings import entity_types, relationship_types

print("所有模块导入成功")

所有模块导入成功


---
## Stage 0：准备 — 读取文件 + 分块

In [2]:
TEST_FILE = str(FILES_DIR / "test.md")
print(f"测试文件: {TEST_FILE}")

reader = FileReader()
docs = reader.read_files(file_paths=TEST_FILE)
doc = docs[0]

chunker = ChineseTextChunker(chunk_size=500, overlap=100)
chunks = chunker.chunk_document(doc)
print(f"Chunk 数量: {len(chunks)}")
if chunks:
    print(f"首个 Chunk: {chunks[0]}")

测试文件: F:\AllProjects\Agent\equipment-fault-qa\data\test\test.md
处理文件: test.md (类型: .md)


Chunk 数量: 2
首个 Chunk: Chunk(
  file_name : F:\AllProjects\Agent\equipment-fault-qa\data\test\test.md
  chunk_id  : 89a1e97ba75e4ad7b8ba628dcf1749dd8be83481
  doc_id    : 338b4ea778e05d0b3cea35a4156c0ea8e910ca20e183f614ee28a8209d9f48b0
  content   : 设备故障分析报告
报告编号:FA-2026-001
报告日期:2026-01-15
报告出处:生产车间一区
制定人:张伟
设备名称:数控加工中心（型号:VMC-...
)


---
## Stage 1：图结构构建 — structure/struct_builder

In [7]:
builder = GraphStructureBuilder()

builder.create_document(
    type=doc.file_name.split(".")[-1],
    uri=TEST_FILE,
    file_name=doc.file_name,
    domain="test"
)
print(f"Document 已创建: {doc.file_name}")

result = builder.create_relation_between_chunks(chunks)
print(f"Chunk 写入完成: {len(result)} 个")

ConfigurationError: URI scheme '' is not supported. Supported URI schemes are ['bolt', 'bolt+ssc', 'bolt+s', 'neo4j', 'neo4j+ssc', 'neo4j+s']. Examples: bolt://host[:port] or neo4j://host[:port][?routing_context]

In [ ]:
graph = get_connection_manager().get_connection()

# ── Stage 1 测试 ──
print("=" * 50)
print("[Stage 1 测试] 图结构写入")
print("=" * 50)

# 1. Document 节点
r = graph.query("MATCH (d:__Document__) RETURN count(d) AS cnt")[0]
assert r['cnt'] == 1, f"Document 数量应为 1，实际 {r['cnt']}"
print(f"  ✓ Document 节点: {r['cnt']} 个")

# 2. Chunk 节点
r = graph.query("MATCH (c:__Chunk__) RETURN count(c) AS cnt")[0]
assert r['cnt'] == len(chunks), f"Chunk 数量应为 {len(chunks)}，实际 {r['cnt']}"
print(f"  ✓ Chunk 节点: {r['cnt']} 个（期望 {len(chunks)}）")

# 3. PART_OF 关系
r = graph.query("""
    MATCH (c:__Chunk__)-[:PART_OF]->(d:__Document__)
    RETURN count(c) AS cnt
""")[0]
assert r['cnt'] == len(chunks), f"PART_OF 应 {len(chunks)} 条，实际 {r['cnt']}"
print(f"  ✓ PART_OF 关系: {r['cnt']} 条")

# 4. FIRST_CHUNK 关系
r = graph.query("""
    MATCH (d:__Document__)-[:FIRST_CHUNK]->(c:__Chunk__)
    RETURN count(c) AS cnt
""")[0]
assert r['cnt'] == 1, f"FIRST_CHUNK 应 1 条，实际 {r['cnt']}"
print(f"  ✓ FIRST_CHUNK 关系: {r['cnt']} 条")

# 5. NEXT_CHUNK 关系
r = graph.query("""
    MATCH (c1:__Chunk__)-[:NEXT_CHUNK]->(c2:__Chunk__)
    RETURN count(c1) AS cnt
""")[0]
assert r['cnt'] == len(chunks) - 1, f"NEXT_CHUNK 应 {len(chunks)-1} 条，实际 {r['cnt']}"
print(f"  ✓ NEXT_CHUNK 关系: {r['cnt']} 条（期望 {len(chunks)-1}）")

# 6. Chunk 属性完整
r = graph.query("""
    MATCH (c:__Chunk__) WHERE c.text IS NULL RETURN count(c) AS cnt
""")[0]
assert r['cnt'] == 0, f"有 {r['cnt']} 个 Chunk 的 text 为空"
print(f"  ✓ 所有 Chunk 属性完整")

print("  ✓ Stage 1 测试通过")

---
## Stage 2：实体关系提取 — extraction/

In [ ]:
# 组装新格式: [(file_name, [Chunk, ...]), ...]
file_contents = [(doc.file_name, chunks)]
print(f"组装完成: {len(chunks)} 个 chunk")

In [ ]:
llm = get_llm_model()

extractor = EntityRelationExtractor(
    llm=llm,
    system_template=system_template_build_graph,
    human_template=human_template_build_graph,
    entity_types=entity_types,
    relationship_types=relationship_types,
    cache_dir=str(PROJECT_ROOT / "cache" / "graph"),
)
print(f"提取器就绪: {len(entity_types)} 种实体类型")

# 执行提取
file_contents = extractor.process_chunks(file_contents)

_, _, extractions = file_contents[0]
non_empty = sum(1 for e in extractions if e.strip())
print(f"提取完成: {len(extractions)} 个 chunk, {non_empty} 个有结果")
if extractions and extractions[0].strip():
    print(f"LLM 输出示例:\n{extractions[0][:300]}")

In [ ]:
writer = GraphWriter()
writer.process_and_write_graph_documents(file_contents)
print("GraphWriter 写入完成")

In [ ]:
# ── Stage 2 测试 ──
print("=" * 50)
print("[Stage 2 测试] 实体关系提取 + 写入")
print("=" * 50)

# 1. LLM 提取结果非空
assert non_empty > 0, "LLM 提取结果全部为空，提取可能失败"
print(f"  ✓ LLM 提取: {non_empty}/{len(extractions)} 个 chunk 有结果")

# 2. Entity 节点
r = graph.query("""
    MATCH (e:__Entity__) RETURN count(e) AS cnt, count(DISTINCT e.type) AS types
""")[0]
assert r['cnt'] > 0, "Entity 节点数量为 0"
print(f"  ✓ __Entity__ 节点: {r['cnt']} 个（{r['types']} 种类型）")

# 3. Entity 必须有 id
r = graph.query("""
    MATCH (e:__Entity__) WHERE e.id IS NULL RETURN count(e) AS cnt
""")[0]
assert r['cnt'] == 0, f"有 {r['cnt']} 个 Entity 缺少 id"
print(f"  ✓ 所有 Entity 均有 id")

# 4. MENTIONS 关系
r = graph.query("""
    MATCH (c:__Chunk__)-[:MENTIONS]->(e:__Entity__)
    RETURN count(c) AS cnt
""")[0]
print(f"  ✓ MENTIONS 关系: {r['cnt']} 条")

# 5. 实体关系
r = graph.query("""
    MATCH (e1:__Entity__)-[r]->(e2:__Entity__)
    RETURN type(r) AS rel_type, count(r) AS cnt
    ORDER BY cnt DESC
    LIMIT 5
""")
entity_rels = sum(row['cnt'] for row in r) if r else 0
if r:
    print(f"  ✓ 实体间关系: {entity_rels} 条")
    for row in r:
        print(f"      {row['rel_type']}: {row['cnt']}")

# 6. 检查有没有解析失败的格式
bad_parse = sum(1 for e in extractions if e.strip() and '"entity"' not in e and '"relationship"' not in e)
if bad_parse:
    print(f"  ⚠ {bad_parse} 个 chunk 解析结果可能不是标准格式")

print("  ✓ Stage 2 测试通过")

---
## Stage 3：向量索引 — indexing/

In [ ]:
print("ChunkIndexManager.create_chunk_index()...")
chunk_indexer = ChunkIndexManager()
chunk_vector = chunk_indexer.create_chunk_index()

print("EntityIndexManager.create_entity_index()...")
entity_indexer = EntityIndexManager()
entity_vector = entity_indexer.create_entity_index()

In [ ]:
# ── Stage 3 测试 ──
print("=" * 50)
print("[Stage 3 测试] 向量索引")
print("=" * 50)

# 1. Chunk embedding
r = graph.query("""
    MATCH (c:__Chunk__) WHERE c.embedding IS NOT NULL RETURN count(c) AS cnt
""")[0]
assert r['cnt'] > 0, f"有 embedding 的 Chunk 为 0"
chunk_embed_dim = graph.query("""
    MATCH (c:__Chunk__) WHERE c.embedding IS NOT NULL
    RETURN size(c.embedding) AS dim LIMIT 1
""")[0]['dim']
print(f"  ✓ Chunk embedding: {r['cnt']}/{len(chunks)} 个, 维度={chunk_embed_dim}")
assert chunk_embed_dim > 0, "Chunk embedding 维度为 0"

# 2. Entity embedding
r = graph.query("""
    MATCH (e:__Entity__) WHERE e.embedding IS NOT NULL RETURN count(e) AS cnt
""")[0]
entity_total = graph.query("MATCH (e:__Entity__) RETURN count(e) AS cnt")[0]['cnt']
if r['cnt'] > 0:
    entity_embed_dim = graph.query("""
        MATCH (e:__Entity__) WHERE e.embedding IS NOT NULL
        RETURN size(e.embedding) AS dim LIMIT 1
    """)[0]['dim']
    print(f"  ✓ Entity embedding: {r['cnt']}/{entity_total} 个, 维度={entity_embed_dim}")
else:
    print(f"  ⚠ Entity embedding: {r['cnt']}/{entity_total} 个（Entity 可能已在索引前被合并）")

# 3. 向量索引是否在 Neo4j 中注册
r = graph.query("""
    SHOW INDEXES YIELD name, type WHERE type = 'VECTOR'
    RETURN name, type
""")
if r:
    for idx in r:
        print(f"  ✓ 向量索引: {idx['name']} ({idx['type']})")

print("  ✓ Stage 3 测试通过")

---
## Stage 4：实体去重 — processing/

需要 Neo4j GDS 插件 (`graph-data-science`)。如果未安装，会捕获异常并跳过。

In [ ]:
try:
    detector = SimilarEntityDetector()
    duplicates, stats = detector.process_entities()
    print(f"检测完成: {len(duplicates)} 组候选重复实体")
    GDS_AVAILABLE = True
except Exception as e:
    print(f"GDS 不可用（跳过 Stage 4a）: {e}")
    duplicates = []
    GDS_AVAILABLE = False

In [ ]:
# ── Stage 4a 测试 ──
print("=" * 50)
print("[Stage 4a 测试] GDS 相似实体检测")
print("=" * 50)

if not GDS_AVAILABLE:
    print("  ⏭ GDS 不可用，跳过测试")
else:
    assert stats['status'] == 'success', f"process_entities 失败: {stats}"
    print(f"  ✓ KNN 关系写入: {stats.get('relationshipsWritten', 0)} 条")
    print(f"  ✓ WCC 社区: {stats.get('communityCount', 0)} 个")
    print(f"  ✓ 候选重复组: {len(duplicates)} 组")
    
    for i, g in enumerate(duplicates[:5]):
        print(f"     组 {i+1}: {g}")
    if len(duplicates) > 5:
        print(f"     ... 还有 {len(duplicates)-5} 组")

    print("  ✓ Stage 4a 测试通过")

In [ ]:
if GDS_AVAILABLE and duplicates:
    merger = EntityMerger()
    merged_count, merge_stats = merger.process_duplicates(duplicates)
    print(f"合并执行完成: {merged_count} 个实体")
elif GDS_AVAILABLE:
    print("没有候选重复实体，跳过合并")
else:
    print("GDS 不可用，跳过合并")

In [ ]:
# ── Stage 4b 测试 ──
print("=" * 50)
print("[Stage 4b 测试] LLM 实体合并")
print("=" * 50)

if not GDS_AVAILABLE:
    print("  ⏭ GDS 不可用，跳过测试")
elif not duplicates:
    print("  ⏭ 无候选组，跳过测试")
else:
    print(f"  ✓ LLM 分析完成: {merge_stats.get('识别出的合并组数', 0)} 组可合并")
    print(f"  ✓ apoc.refactor.mergeNodes 执行: {merged_count} 个实体被合并")
    
    r = graph.query("MATCH (e:__Entity__) RETURN count(e) AS cnt")[0]['cnt']
    print(f"  ✓ 合并后 Entity 总量: {r} 个")

    print("  ✓ Stage 4b 测试通过")

---
## Stage 5：实体质量提升 — processing/entity_quality

消歧（设 canonical_id）+ 对齐（合并同一 canonical 的实体）。

In [ ]:
try:
    quality = EntityQualityProcessor()
    quality_result = quality.process()
    QUALITY_OK = True
except Exception as e:
    print(f"质量提升失败: {e}")
    QUALITY_OK = False

In [ ]:
# ── Stage 5 测试 ──
print("=" * 50)
print("[Stage 5 测试] 消歧 + 对齐")
print("=" * 50)

if not QUALITY_OK:
    print("  ⏭ 质量提升未执行，跳过测试")
else:
    # 1. 消歧
    print(f"  ✓ 消歧: {quality_result['disambiguated']} 个实体设了 canonical_id")
    
    r = graph.query("""
        MATCH (e:__Entity__)
        WHERE e.canonical_id IS NOT NULL
        RETURN count(e) AS cnt
    """)[0]['cnt']
    if r > 0:
        print(f"  ✓ Neo4j 中 canonical_id 不为空的 Entity: {r} 个")
    else:
        print(f"  ⚠ canonical_id 未设置（可能所有实体各有独立 ID，无需消歧）")
    
    # 2. 对齐
    print(f"  ✓ 对齐: {quality_result['aligned']} 个实体被合并")
    
    r = graph.query("""
        MATCH (e:__Entity__)
        WHERE e.aligned_from IS NOT NULL
        RETURN count(e) AS cnt
    """)[0]['cnt']
    if r > 0:
        print(f"  ✓ 对齐后的主实体（aligned_from 不为空）: {r} 个")
    
    print("  ✓ Stage 5 测试通过")

---
## Stage 6：一致性校验

In [ ]:
validator = GraphConsistencyValidator()
validator.display_graph_stats()
validation_result = validator.process(repair=True)

In [ ]:
# ── Stage 6 测试 ──
print("=" * 50)
print("[Stage 6 测试] 一致性校验")
print("=" * 50)

vr = validation_result
assert vr.get('validation_time', 0) > 0, "验证未执行"
print(f"  ✓ 验证耗时: {vr['validation_time']:.2f}秒")

vs = vr.get('validation_stats', {})
total_issues = vs.get('total_issues', 0)
print(f"  ✓ 发现的问题: {total_issues} 个")
for k, v in vs.items():
    if k != 'total_issues':
        print(f"     {k}: {v}")

if 'repair_result' in vr:
    repairs = vr['repair_result'].get('repairs', {})
    total_repaired = sum(repairs.values())
    print(f"  ✓ 已修复数: {total_repaired} 个")
    for k, v in repairs.items():
        print(f"     {k}: {v}")

print("  ✓ Stage 6 测试通过")

---
## 全链路汇总

In [ ]:
print("\n" + "=" * 50)
print("图谱最终统计")
print("=" * 50)

r = graph.query("""MATCH (n) RETURN labels(n) AS label, count(n) AS cnt ORDER BY cnt DESC""")
node_counts = {}
for row in r:
    for lbl in row['label']:
        node_counts[lbl] = node_counts.get(lbl, 0) + row['cnt']
for lbl, cnt in sorted(node_counts.items(), key=lambda x: -x[1]):
    print(f"  {lbl}: {cnt}")

r = graph.query("""MATCH ()-[r]->() RETURN type(r) AS t, count(r) AS cnt ORDER BY cnt DESC""")
print()
for row in r:
    print(f"  {row['t']}: {row['cnt']}")

In [ ]:
# ── 全链路测试汇总 ──
print("\n" + "=" * 50)
print("测试汇总")
print("=" * 50)

summary = [
    ("Stage 0 文件读取+分块", True),
    ("Stage 1 图结构构建", True),
    ("Stage 2 实体提取+写入", non_empty > 0),
    ("Stage 3 向量索引", chunk_embed_dim > 0),
    ("Stage 4a GDS 相似检测", not GDS_AVAILABLE or stats['status'] == 'success'),
    ("Stage 4b LLM 实体合并", not GDS_AVAILABLE or not duplicates or merged_count >= 0),
    ("Stage 5 消歧+对齐", not QUALITY_OK or quality_result['disambiguated'] >= 0),
    ("Stage 6 一致性校验", True),
]

all_pass = True
for name, passed in summary:
    icon = "✓" if passed else "✗"
    if not passed:
        all_pass = False
    print(f"  [{icon}] {name}")

print(f"\n{'全部通过!' if all_pass else '有失败项，请检查上方输出'}")

---
## 清理（可选）

In [ ]:
# builder.clear_database()
# print("数据库已清空")